In [1]:
import os
# os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

import jax
import jax.numpy as jnp
import numpy as np
import dill

In [2]:
res = dill.load(open("/home/bryanpu1/projects/parallel_vs_serial/scaling_jax/src/evaluations/res-metastable_ppo-scratch_1M-0_cot_tokens-8x8-11-18-25_09_03_56-deacc19b-7394-432c-aca2-8d5d3592c247-temp_1.0-decode_len_100-num_evals_32.dill", "rb"))

In [3]:
batch = res["results"][20][8]

In [4]:
batch.keys()

dict_keys(['actions', 'eos', 'last_prompt_idx', 'logits', 'observations', 'question_mask'])

In [5]:
sample_i = 0

In [6]:
question_mask = np.ones(batch["observations"].shape[-1])
question_mask[batch["last_prompt_idx"][sample_i] + 1:] = 0
answer_mask = 1 - batch["question_mask"][sample_i]

In [7]:
pred_mask = np.zeros(batch["observations"].shape[-1])
pred_mask[batch["last_prompt_idx"][sample_i]:] = 1

In [8]:
response = np.concatenate((
    batch["observations"][sample_i][np.where(question_mask)],
    batch["actions"][sample_i][np.where(answer_mask)],
))

In [9]:
first_bin_repr = "".join(map(str, batch["observations"][sample_i][:8]))
second_bin_repr = "".join(map(str, batch["observations"][sample_i][9:17]))

print(first_bin_repr)
print(second_bin_repr)

carry = False
soln_bin_repr = ""
for first_bit, second_bit in zip(
    first_bin_repr,
    second_bin_repr,
):
    curr_res = int(first_bit) + int(second_bit) + carry
    carry = curr_res >= 2
    soln_bin_repr = soln_bin_repr + str(
        curr_res % 2
    )
print(soln_bin_repr)

target = soln_bin_repr
eos_token_id = 6

11110110
01010001
10011111


In [10]:
def get_success(response, target, mask):
    # XXX: Stop at first matching string
    response = "".join(np.array(response).astype(str))
    success = float(target in response)

    assert str(eos_token_id) not in response

    if success:
        end_idx = response.find(target) + len(target) - 1
        mask[end_idx:] = 0

    # print(success, target, response)
    response_length = np.sum(mask)

    has_eos = 1.0
    return success, response_length, has_eos, mask

success, response_length, curr_has_eos, pred_mask = get_success(
    response,
    target,
    pred_mask,
)
reward = (-1) ** (1 - success)

In [11]:
success, response_length, curr_has_eos, pred_mask

(1.0,
 np.float64(36.0),
 1.0,
 array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]))

In [12]:
reset_token_id = 3

@jax.jit
def scan_fn(carry, idx):
    pointer_correct = carry["pointer_correct"]
    actions = carry["actions"]
    target = carry["target"]
    pred_mask = carry["pred_mask"]
    last_reset_idx = carry["last_reset_idx"]
    first_mistake_idx = carry["first_mistake_idx"]
    reset_idxes = carry["reset_idxes"]
    first_mistake_idxes = carry["first_mistake_idxes"]
    curr_trial = carry["curr_trial"]

    action_match = actions[idx] == target[pointer_correct]
    is_reset = actions[idx] == reset_token_id
    is_reset_with_pred = jnp.logical_and(is_reset, pred_mask[idx])

    # Shift the pointer if the action matches the target and we're within a prediction mask
    reset_pointer = jax.lax.select(
        is_reset,
        1,
        0,
    )
    pointer_correct = jax.lax.select(
        pred_mask[idx],
        jax.lax.select(
            action_match,
            pointer_correct + 1,
            reset_pointer,
        ),
        pointer_correct,
    )

    # Update the last reset index to current index upon new trial
    last_reset_idx = jax.lax.select(
        is_reset_with_pred,
        idx,
        last_reset_idx,
    )

    reset_idxes = reset_idxes.at[curr_trial + 1].set(
        jax.lax.select(
            is_reset_with_pred,
            last_reset_idx,
            reset_idxes[curr_trial + 1],
        )
    )
    first_mistake_idxes = first_mistake_idxes.at[curr_trial].set(first_mistake_idx)

    # Identify the first mistake index within the current trial
    first_mistake_idx = jax.lax.select(
        jnp.logical_or(action_match, is_reset),
        idx + 1,
        jax.lax.select(
            first_mistake_idx > reset_idxes[curr_trial],
            first_mistake_idx,
            idx,
        ),
    )

    curr_trial = jax.lax.select(
        is_reset_with_pred,
        curr_trial + 1,
        curr_trial,
    )

    return {
        "pointer_correct": pointer_correct,
        "last_reset_idx": last_reset_idx,
        "first_mistake_idx": first_mistake_idx,
        "actions": actions,
        "target": target,
        "pred_mask": pred_mask,
        "reset_idxes": reset_idxes,
        "first_mistake_idxes": first_mistake_idxes,
        "curr_trial": curr_trial,
    }, None

def process_reward(batch, rewards, response_lengths, has_eos):
    actions = batch["actions"]
    target = batch["target"]
    mask = batch["pred_mask"]

    pointer_correct = np.array(1, dtype=int)
    last_reset_idx = np.array(-1, dtype=int)
    first_mistake_idx = np.array(-1, dtype=int)
    curr_trial = np.array(0, dtype=int)
    reset_idxes = np.full_like(actions, fill_value=-1, dtype=int)
    reset_idxes[0] = np.where(mask == 1)[0][0] - 1
    first_mistake_idxes = np.full_like(actions, fill_value=-1, dtype=int)
    last_idx = min(np.where(mask == 1)[0][-1] + 1, actions.shape[-1])

    res, _ = jax.lax.scan(
        scan_fn,
        {
            "pointer_correct": pointer_correct,
            "last_reset_idx": last_reset_idx,
            "first_mistake_idx": first_mistake_idx,
            "actions": actions,
            "target": target,
            "pred_mask": mask.astype(int),
            "reset_idxes": reset_idxes,
            "first_mistake_idxes": first_mistake_idxes,
            "curr_trial": curr_trial,
        },
        np.arange(last_idx),
    )

    return res


In [13]:
batch["last_prompt_idx"][0]

Array(17, dtype=int32)

In [14]:
np.where(pred_mask == 1)[0]

array([17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52])

In [30]:
res = process_reward(
    {
        "observations": batch["observations"][sample_i],
        "actions": batch["actions"][sample_i].at[25:].set(0),
        "target": np.array([3] + [int(el) for el in target] + [eos_token_id] * (batch["observations"][sample_i].shape[-1] - len(target) - 1)),
        "pred_mask": jnp.array(pred_mask).at[25:].set(1),
    },
    reward,
    response_length,
    curr_has_eos,
)

In [31]:
res

{'actions': Array([0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1,
        0, 1, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32),
 'curr_trial': Array(1, dtype=int32),
 'first_mistake_idx': Array(25, dtype=int32),
 'first_mistake_idxes': Array([18, 25, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],      dtype=int32),
 'last_reset_idx': Array(24, dtype=int32),
 'pointer

In [32]:
pred_mask = res["pred_mask"]
reset_idxes = res["reset_idxes"]
first_mistake_idxes = res["first_mistake_idxes"]
actions = res["actions"]

last_idx = min(np.where(pred_mask == 1)[0][-1] + 1, actions.shape[-1])
correct_lens = np.concatenate(([0], first_mistake_idxes - reset_idxes))
improvements = correct_lens[1:] - correct_lens[:-1]
reset_idxes = reset_idxes.at[(np.where(reset_idxes == -1))[0][0]].set(last_idx)
trial_lengths = np.diff(reset_idxes[reset_idxes != -1])
returns = np.full_like(actions, fill_value=-1)
returns[reset_idxes[0]:last_idx] = np.repeat(improvements[:int(np.sum(reset_idxes != -1)) - 1], trial_lengths)

In [33]:

print(correct_lens)
print(improvements)

[0 2 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[ 2 -1 -1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0]


In [34]:
returns

array([-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  2,
        2,  2,  2,  2,  2,  2,  2, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
      dtype=int32)